# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 3 — Multi-split benchmark, partitioning cost, and bond-feature ablation

This notebook evaluates all five models across fifteen independent scaffold partitions and
fifteen random partitions of identical proportions, and isolates the contribution of bond
information through an ablation in which the message-passing layers are replaced by
variants that ignore edge attributes.

Fifteen partitions give a Nemenyi critical difference of 1.58 rank positions for five
models, which is small enough for a difference of practical size to be resolved. Every
partition is evaluated with the same protocol used on the primary split: three seeds per
model, a decision threshold chosen on the validation set, and a temperature fitted on the
validation set and applied to the test set.

**Contents**

1. Environment and shared definitions
2. Data and configurations
3. Evaluation routine
4. Hyperparameter search for the ablation variants
5. Scaffold benchmark
6. Bond-feature ablation
7. Random-partition benchmark
8. Statistical comparison

*Approximate runtime: 10-15 hours on a T4. Every completed model is written to disk, and
re-running the notebook resumes from the last completed item.*

## 1. Environment and shared definitions

In [ ]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q optuna
!pip install -q scikit-posthocs
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK    = "/content/drive/MyDrive/ames_pipeline"
DATADIR = f"{WORK}/data"
OUTDIR  = f"{WORK}/results"
CKPT    = f"{WORK}/ckpt"
FIGDIR  = f"{WORK}/figures"
MODELS  = f"{WORK}/models"
for d in (WORK, DATADIR, OUTDIR, CKPT, FIGDIR, MODELS):
    os.makedirs(d, exist_ok=True)
print("Working directory:", WORK)

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ames_pipeline


In [ ]:
%%writefile /content/ames_lib.py
"""Shared definitions for the Ames mutagenicity pipeline."""

import random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# ----------------------------- partitioning -----------------------------

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try:
            return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception:
            return smi


def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs  = list(groups.values())
    rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d)
    n_train, n_train_valid = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr) + len(g) <= n_train:
            tr += g
        elif len(tr) + len(va) + len(g) <= n_train_valid:
            va += g
        else:
            te += g
    return d.loc[tr], d.loc[va], d.loc[te]


def make_random_split(d, seed, frac=(0.8, 0.1, 0.1)):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(d.index.values)
    n = len(d)
    a, b = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    return d.loc[idx[:a]], d.loc[idx[a:b]], d.loc[idx[b:]]


# ---------------------------- representations ----------------------------

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr


def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None:
            X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)


def one_hot(x, allowable):
    if x not in allowable:
        x = allowable[-1]
    return [int(x == a) for a in allowable]


def atom_features(atom):
    f  = one_hot(atom.GetSymbol(), ATOM_LIST)
    f += one_hot(str(atom.GetHybridization()), HYBRIDIZATION)
    f += [atom.GetDegree(), atom.GetFormalCharge(), int(atom.GetIsAromatic()),
          atom.GetTotalNumHs(), int(atom.IsInRing())]
    return f


def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(bond.GetIsConjugated()), int(bond.IsInRing())]


def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        ei += [[i, j], [j, i]]
        ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)


def build_dataset(d):
    graphs = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [g for g in graphs if g is not None]


# ------------------------------ architectures ------------------------------

class GNN(nn.Module):
    """Message-passing classifier.

    GINE and GAT consume bond features; GCN operates on connectivity alone.
    Setting use_edge_attr=False substitutes GIN for GINE and removes the edge
    channel from GAT, which isolates the contribution of bond information.
    """

    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn":
                conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else:
                raise ValueError(f"Unknown architecture: {kind}")
            self.convs.append(conv)
            self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, data):
        x, edge_index, edge_attr, batch = (data.x, data.edge_index,
                                           data.edge_attr, data.batch)
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = (conv(x, edge_index, edge_attr) if self.use_edge_attr
                 else conv(x, edge_index))
            h = F.relu(bn(h))
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, batch), global_add_pool(x, batch)], dim=1)
        return self.head(g).squeeze(-1)


def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4,
              use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads,
               use_edge_attr=use_edge_attr).to(DEVICE)


# ------------------------------- training -------------------------------

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, logits = [], [], []
    for d in loader:
        d = d.to(DEVICE)
        logit = model(d)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(logit).cpu().numpy())
        logits.append(logit.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(logits)


def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, trial=None, use_edge_attr=True):
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    train_loader = DataLoader(tr_set, batch_size=params["batch_size"],
                              shuffle=True, drop_last=True)
    valid_loader = DataLoader(va_set, batch_size=256)

    best_auc, best_state, waited = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for d in train_loader:
            d = d.to(DEVICE)
            optimiser.zero_grad()
            criterion(model(d), d.y.view(-1)).backward()
            optimiser.step()
        y_va, p_va, _ = evaluate(model, valid_loader)
        auc = roc_auc_score(y_va, p_va) if len(np.unique(y_va)) > 1 else 0.5
        if trial is not None:
            trial.report(auc, epoch)
            if trial.should_prune():
                import optuna
                raise optuna.TrialPruned()
        if auc > best_auc:
            best_auc, waited = auc, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_auc


# ------------------------------ calibration ------------------------------

def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def fit_temperature(logits, y):
    z = torch.tensor(logits, dtype=torch.float32)
    t = torch.tensor(y, dtype=torch.float32)
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=200)
    lossf = torch.nn.BCEWithLogitsLoss()
    def closure():
        opt.zero_grad()
        loss = lossf(z / torch.exp(log_T), t)
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.exp(log_T).item())


def ece_equal_width(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e

Writing /content/ames_lib.py


In [ ]:
import sys, json, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames_lib import *
from torch_geometric.loader import DataLoader

print("Device:", DEVICE, "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Device: cuda | Tesla T4


## 2. Data and configurations

The compound pool is rebuilt from the source tables. The configurations selected in Part 1
are reused for every partition: the search was carried out once, on the primary partition,
and the resulting configuration is applied unchanged elsewhere. This keeps the comparison
across partitions free of selection effects that would otherwise differ from split to
split.

In [ ]:
SEED                 = 42
SEEDS                = [42, 43, 44]
PRIMARY_SPLIT_SEED   = 101
SCAFFOLD_SPLIT_SEEDS = list(range(42, 57))
RANDOM_SPLIT_SEEDS   = list(range(42, 57))
MAX_EPOCHS, PATIENCE = 250, 30
N_TRIALS_ABLATION    = 100
RUN_ABLATION         = True

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}")
                for f in ["train.csv", "valid.csv", "test.csv"]],
               ignore_index=True)[["Drug_ID", "smiles", "label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)

HP_CLASSIC = json.load(open(f"{OUTDIR}/hpo_classic.json"))
HP_GNN     = json.load(open(f"{OUTDIR}/hpo_gnn.json"))

print(f"{len(df)} compounds")
print(f"{len(SCAFFOLD_SPLIT_SEEDS)} scaffold and {len(RANDOM_SPLIT_SEEDS)} random partitions")

7255 compounds
15 scaffold and 15 random partitions


## 3. Evaluation routine

One function fits and scores every model on a given partition. Results are appended to a
single table on disk, keyed by partition and model, and any combination already present is
skipped. This makes the benchmark restartable at the granularity of a single model.

In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, matthews_corrcoef,
                             recall_score, accuracy_score)

RUNS_PATH = f"{OUTDIR}/benchmark_runs.csv"

def load_runs():
    if os.path.exists(RUNS_PATH):
        return pd.read_csv(RUNS_PATH)
    return pd.DataFrame(columns=["partition", "scheme", "seed", "model", "arm"])

def append_run(row):
    runs = load_runs()
    runs = pd.concat([runs, pd.DataFrame([row])], ignore_index=True)
    runs.to_csv(RUNS_PATH, index=False)

def done(partition, model):
    runs = load_runs()
    return ((runs["partition"] == partition) & (runs["model"] == model)).any()

def best_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    return grid[int(np.argmax([balanced_accuracy_score(y, (p >= t).astype(int))
                               for t in grid]))]

def score_model(name, arm, partition, scheme, split_seed,
                y_valid, p_valid, y_test, p_test):
    thr  = best_threshold(y_valid, p_valid)
    yhat = (p_test >= thr).astype(int)
    T    = fit_temperature(to_logit(p_valid), y_valid)
    p_cal = 1 / (1 + np.exp(-to_logit(p_test) / T))
    return {
        "partition": partition, "scheme": scheme, "seed": split_seed,
        "model": name, "arm": arm, "threshold": thr,
        "ROC_AUC":      roc_auc_score(y_test, p_test),
        "PR_AUC":       average_precision_score(y_test, p_test),
        "accuracy":     accuracy_score(y_test, yhat),
        "balanced_acc": balanced_accuracy_score(y_test, yhat),
        "sensitivity":  recall_score(y_test, yhat, pos_label=1),
        "specificity":  recall_score(y_test, yhat, pos_label=0),
        "MCC":          matthews_corrcoef(y_test, yhat),
        "temperature":  T,
        "ECE_raw":      ece_equal_width(y_test, p_test, 10),
        "ECE_calibrated": ece_equal_width(y_test, p_cal, 10),
        "test_positive_rate": float(y_test.mean()),
    }

def run_partition(partition, scheme, split_seed, spec):
    """spec: list of (display_name, arm, kind_or_None, hp, use_edge_attr)."""
    if scheme == "scaffold":
        d_tr, d_va, d_te = make_scaffold_split(df, seed=split_seed)
    else:
        d_tr, d_va, d_te = make_random_split(df, seed=split_seed)
    d_tr, d_va, d_te = [x.reset_index(drop=True) for x in (d_tr, d_va, d_te)]

    need_classic = any(s[2] is None for s in spec)
    need_graph   = any(s[2] is not None for s in spec)

    if need_classic:
        Xtr, d_tr2 = build_X(d_tr); ytr = d_tr2["label"].values
        Xva, d_va2 = build_X(d_va); yva = d_va2["label"].values
        Xte, d_te2 = build_X(d_te); yte = d_te2["label"].values
        spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)

    if need_graph:
        tr_set, va_set, te_set = (build_dataset(d_tr), build_dataset(d_va),
                                  build_dataset(d_te))
        va_loader = DataLoader(va_set, batch_size=256)
        te_loader = DataLoader(te_set, batch_size=256)
        yg = d_tr["label"].values
        pw = torch.tensor([(yg == 0).sum() / max((yg == 1).sum(), 1)],
                          dtype=torch.float, device=DEVICE)

    for name, arm, kind, hp, use_edge in spec:
        if done(partition, name):
            continue
        t0 = time.time()
        if kind is None:
            pv, pt = [], []
            for s in SEEDS:
                if name.startswith("XGBoost"):
                    m = XGBClassifier(n_estimators=1500, **hp, scale_pos_weight=spw,
                                      eval_metric="auc", early_stopping_rounds=50,
                                      tree_method="hist", random_state=s, n_jobs=-1)
                    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
                else:
                    m = RandomForestClassifier(**hp, class_weight="balanced",
                                               random_state=s, n_jobs=-1)
                    m.fit(Xtr, ytr)
                pv.append(m.predict_proba(Xva)[:, 1])
                pt.append(m.predict_proba(Xte)[:, 1])
            row = score_model(name, arm, partition, scheme, split_seed,
                              yva, np.mean(pv, 0), yte, np.mean(pt, 0))
        else:
            pv, pt = [], []
            for s in SEEDS:
                model, _ = train_gnn(kind, hp, s, tr_set, va_set, pw,
                                     MAX_EPOCHS, PATIENCE, use_edge_attr=use_edge)
                yva_g, p_va, _ = evaluate(model, va_loader)
                yte_g, p_te, _ = evaluate(model, te_loader)
                pv.append(p_va); pt.append(p_te)
                del model; torch.cuda.empty_cache()
            row = score_model(name, arm, partition, scheme, split_seed,
                              yva_g, np.mean(pv, 0), yte_g, np.mean(pt, 0))
        row["wall_clock_min"] = (time.time() - t0) / 60
        append_run(row)
        print(f"  {partition:16s} {name:12s} ROC-AUC {row['ROC_AUC']:.4f}  "
              f"({row['wall_clock_min']:.1f} min)")

## 4. Hyperparameter search for the ablation variants

The ablated architectures are searched independently, over the same space and with the same
trial budget as their counterparts. Reusing a configuration selected with bond features
enabled would confound the contribution of bond information with the suitability of the
configuration, so each variant is given its own search.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

if RUN_ABLATION:
    d_tr, d_va, _ = [x.reset_index(drop=True)
                     for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]
    tr_set_p, va_set_p = build_dataset(d_tr), build_dataset(d_va)
    yg = d_tr["label"].values
    pw_p = torch.tensor([(yg == 0).sum() / max((yg == 1).sum(), 1)],
                        dtype=torch.float, device=DEVICE)

    def make_objective_noedge(kind):
        def objective(trial):
            params = {
                "hidden":       trial.suggest_categorical("hidden", [64, 128, 256]),
                "n_layers":     trial.suggest_int("n_layers", 2, 5),
                "dropout":      trial.suggest_float("dropout", 0.0, 0.5),
                "lr":           trial.suggest_float("lr", 1e-4, 5e-3, log=True),
                "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
                "batch_size":   trial.suggest_categorical("batch_size", [32, 64, 128]),
                "residual":     trial.suggest_categorical("residual", [True, False]),
            }
            _, auc = train_gnn(kind, params, SEED, tr_set_p, va_set_p, pw_p,
                               MAX_EPOCHS, PATIENCE, trial=trial, use_edge_attr=False)
            return auc
        return objective

    hp_noedge_path = f"{OUTDIR}/hpo_gnn_noedge.json"
    HP_NOEDGE = json.load(open(hp_noedge_path)) if os.path.exists(hp_noedge_path) else {}
    for kind in ["gin", "gat"]:
        if kind in HP_NOEDGE:
            print(f"{kind.upper()} (no edge features): loaded from checkpoint")
            continue
        t0 = time.time()
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=SEED),
            pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
        study.optimize(make_objective_noedge(kind), n_trials=N_TRIALS_ABLATION,
                       show_progress_bar=True)
        HP_NOEDGE[kind] = study.best_params
        json.dump(HP_NOEDGE, open(hp_noedge_path, "w"), indent=2)
        print(f"{kind.upper()} (no edge features) validation ROC-AUC = "
              f"{study.best_value:.4f}  ({(time.time()-t0)/60:.1f} min)")
        print(f"   {study.best_params}")
    del tr_set_p, va_set_p; torch.cuda.empty_cache()
else:
    HP_NOEDGE = {k: HP_GNN[k] for k in ["gin", "gat"]}

GIN (no edge features): loaded from checkpoint
GAT (no edge features): loaded from checkpoint


## 5. Scaffold benchmark

Fifteen scaffold partitions, five models, three seeds each.

In [ ]:
MAIN_SPEC = [
    ("XGBoost",      "main", None,  HP_CLASSIC["XGBoost"],      None),
    ("RandomForest", "main", None,  HP_CLASSIC["RandomForest"], None),
    ("GCN",          "main", "gcn", HP_GNN["gcn"],              True),
    ("GIN",          "main", "gin", HP_GNN["gin"],              True),
    ("GAT",          "main", "gat", HP_GNN["gat"],              True),
]

for sd in SCAFFOLD_SPLIT_SEEDS:
    print(f"scaffold_{sd}")
    run_partition(f"scaffold_{sd}", "scaffold", sd, MAIN_SPEC)

runs = load_runs()
print(f"\nCompleted runs: {len(runs)}")

scaffold_42
scaffold_43
scaffold_44
scaffold_45
scaffold_46
scaffold_47
scaffold_48
scaffold_49
scaffold_50
scaffold_51
scaffold_52
scaffold_53
scaffold_54
scaffold_55
scaffold_56
  scaffold_56      GAT          ROC-AUC 0.7992  (7.0 min)

Completed runs: 75


## 6. Bond-feature ablation

The same fifteen scaffold partitions are evaluated with GIN and GAT variants that ignore
edge attributes. Every other element of the pipeline is held fixed, so the paired
difference measures the contribution of bond information.

In [ ]:
if RUN_ABLATION:
    ABLATION_SPEC = [
        ("GIN_noedge", "ablation", "gin", HP_NOEDGE["gin"], False),
        ("GAT_noedge", "ablation", "gat", HP_NOEDGE["gat"], False),
    ]
    for sd in SCAFFOLD_SPLIT_SEEDS:
        print(f"scaffold_{sd} (ablation)")
        run_partition(f"scaffold_{sd}", "scaffold", sd, ABLATION_SPEC)

    runs = load_runs()
    print(f"\nCompleted runs: {len(runs)}")

scaffold_42 (ablation)
  scaffold_42      GIN_noedge   ROC-AUC 0.7891  (3.2 min)
  scaffold_42      GAT_noedge   ROC-AUC 0.7880  (5.7 min)
scaffold_43 (ablation)
  scaffold_43      GIN_noedge   ROC-AUC 0.8021  (3.7 min)
  scaffold_43      GAT_noedge   ROC-AUC 0.7968  (8.2 min)
scaffold_44 (ablation)
  scaffold_44      GIN_noedge   ROC-AUC 0.8029  (3.1 min)
  scaffold_44      GAT_noedge   ROC-AUC 0.8038  (7.2 min)
scaffold_45 (ablation)
  scaffold_45      GIN_noedge   ROC-AUC 0.8007  (3.2 min)
  scaffold_45      GAT_noedge   ROC-AUC 0.7933  (7.0 min)
scaffold_46 (ablation)
  scaffold_46      GIN_noedge   ROC-AUC 0.8086  (3.5 min)
  scaffold_46      GAT_noedge   ROC-AUC 0.8069  (6.2 min)
scaffold_47 (ablation)
  scaffold_47      GIN_noedge   ROC-AUC 0.8165  (3.3 min)
  scaffold_47      GAT_noedge   ROC-AUC 0.8067  (5.2 min)
scaffold_48 (ablation)
  scaffold_48      GIN_noedge   ROC-AUC 0.8060  (3.4 min)
  scaffold_48      GAT_noedge   ROC-AUC 0.7926  (6.2 min)
scaffold_49 (ablation)
  sc

## 7. Random-partition benchmark

Fifteen random partitions of identical proportions, evaluated with the same five models,
so that the cost of scaffold partitioning is estimated with the same precision as the
scaffold results themselves.

In [10]:
for sd in RANDOM_SPLIT_SEEDS:
    print(f"random_{sd}")
    run_partition(f"random_{sd}", "random", sd, MAIN_SPEC)

runs = load_runs()
print(f"\nCompleted runs: {len(runs)}")

random_42
  random_42        XGBoost      ROC-AUC 0.9017  (0.1 min)
  random_42        RandomForest ROC-AUC 0.9078  (0.3 min)
  random_42        GCN          ROC-AUC 0.9057  (2.9 min)
  random_42        GIN          ROC-AUC 0.9145  (13.6 min)
  random_42        GAT          ROC-AUC 0.9177  (12.7 min)
random_43
  random_43        XGBoost      ROC-AUC 0.8900  (0.1 min)
  random_43        RandomForest ROC-AUC 0.8996  (0.3 min)
  random_43        GCN          ROC-AUC 0.8958  (2.5 min)
  random_43        GIN          ROC-AUC 0.8948  (11.2 min)
  random_43        GAT          ROC-AUC 0.9057  (12.6 min)
random_44
  random_44        XGBoost      ROC-AUC 0.8944  (0.2 min)
  random_44        RandomForest ROC-AUC 0.8983  (0.3 min)
  random_44        GCN          ROC-AUC 0.8987  (2.9 min)
  random_44        GIN          ROC-AUC 0.8974  (11.4 min)
  random_44        GAT          ROC-AUC 0.9090  (14.5 min)
random_45
  random_45        XGBoost      ROC-AUC 0.9017  (0.1 min)
  random_45        RandomF

## 8. Statistical comparison

The five main models are compared across the scaffold partitions with a Friedman test,
followed by a Nemenyi post-hoc test when the omnibus test rejects. The ablation and the
partitioning comparison are paired over partitions and tested with a Wilcoxon signed-rank
test, with the matched-pairs rank-biserial correlation reported as the effect size.

In [11]:
from scipy.stats import friedmanchisquare, wilcoxon, rankdata
import scikit_posthocs as sp

runs = load_runs()
MODEL_ORDER = ["XGBoost", "RandomForest", "GCN", "GIN", "GAT"]
METRICS = ["ROC_AUC", "PR_AUC", "balanced_acc", "sensitivity",
           "specificity", "MCC", "ECE_raw", "ECE_calibrated", "temperature"]

sc = runs[(runs.scheme == "scaffold") & (runs.arm == "main")]
summary = (sc.groupby("model")[METRICS].agg(["mean", "std"])
             .reindex(MODEL_ORDER).round(4))
summary.to_csv(f"{OUTDIR}/table06_scaffold_summary.csv")
print("Scaffold partitions, mean and standard deviation across splits")
print(summary.to_string())

pivot = sc.pivot_table(index="partition", columns="model",
                       values="ROC_AUC").reindex(columns=MODEL_ORDER).dropna()
N, k = pivot.shape
stat, p = friedmanchisquare(*[pivot[m].values for m in MODEL_ORDER])
Q = {3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949}
CD = Q[k] * np.sqrt(k * (k + 1) / (6 * N))
ranks = pd.Series(np.mean([rankdata(-row) for row in pivot.values], axis=0),
                  index=MODEL_ORDER).sort_values()
print(f"\nFriedman on {N} partitions: chi2 = {stat:.3f}, p = {p:.4f}")
print(f"Nemenyi critical difference at alpha = 0.05: {CD:.3f} rank positions")
print("Mean ranks:")
for m, r in ranks.items():
    print(f"  {m:14s} {r:.2f}")
print(f"Observed rank spread: {ranks.max() - ranks.min():.2f}")

nem = sp.posthoc_nemenyi_friedman(pivot.values)
nem.index = nem.columns = MODEL_ORDER
nem.to_csv(f"{OUTDIR}/table07_nemenyi.csv")
pd.DataFrame({"model": ranks.index, "mean_rank": ranks.values}).to_csv(
    f"{OUTDIR}/table07b_mean_ranks.csv", index=False)
print("\nNemenyi pairwise p-values")
print(nem.round(3).to_string())

Scaffold partitions, mean and standard deviation across splits
             ROC_AUC          PR_AUC         balanced_acc         sensitivity         specificity             MCC         ECE_raw         ECE_calibrated         temperature        
                mean     std    mean     std         mean     std        mean     std        mean     std    mean     std    mean     std           mean     std        mean     std
model                                                                                                                                                                               
XGBoost       0.7801  0.0095  0.7612  0.0178       0.7151  0.0089      0.6767  0.0497      0.7535  0.0471  0.4334  0.0174  0.0605  0.0111         0.0640  0.0120      0.9121  0.0935
RandomForest  0.7949  0.0078  0.7784  0.0105       0.7264  0.0080      0.6427  0.0331      0.8101  0.0367  0.4620  0.0177  0.0861  0.0080         0.0796  0.0078      0.6794  0.0350
GCN           0.7909  0.0082  0.

In [12]:
def paired_test(a, b, label_a, label_b):
    """Wilcoxon signed-rank test with a matched-pairs rank-biserial correlation."""
    d = np.asarray(a) - np.asarray(b)
    nz = d[d != 0]
    stat, p = wilcoxon(d)
    r = rankdata(np.abs(nz))
    W_pos, W_neg = r[nz > 0].sum(), r[nz < 0].sum()
    rb = (W_pos - W_neg) / (W_pos + W_neg)
    return {"comparison": f"{label_a} - {label_b}", "n_pairs": len(d),
            "mean_difference": d.mean(), "median_difference": float(np.median(d)),
            "n_improved": int((d > 0).sum()), "n_worsened": int((d < 0).sum()),
            "wilcoxon_p": p, "rank_biserial": rb}

rows = []
if RUN_ABLATION:
    ab = runs[(runs.scheme == "scaffold")]
    for base, abl in [("GIN", "GIN_noedge"), ("GAT", "GAT_noedge")]:
        piv = ab.pivot_table(index="partition", columns="model",
                             values="ROC_AUC")[[base, abl]].dropna()
        if len(piv):
            rows.append(paired_test(piv[base].values, piv[abl].values, base, abl))

sc_mean = sc.groupby("model")["ROC_AUC"].mean()
rn = runs[(runs.scheme == "random") & (runs.arm == "main")]
rn_mean = rn.groupby("model")["ROC_AUC"].mean()
for m in MODEL_ORDER:
    a = rn[rn.model == m].sort_values("seed")["ROC_AUC"].values
    b = sc[sc.model == m].sort_values("seed")["ROC_AUC"].values
    n = min(len(a), len(b))
    if n >= 5:
        rows.append(paired_test(a[:n], b[:n], f"random_{m}", f"scaffold_{m}"))

paired_df = pd.DataFrame(rows)
paired_df.to_csv(f"{OUTDIR}/table08_paired_comparisons.csv", index=False)
print(paired_df.round(4).to_string(index=False))

split_cmp = pd.DataFrame({
    "model": MODEL_ORDER,
    "scaffold_mean": [sc_mean.get(m, np.nan) for m in MODEL_ORDER],
    "random_mean":   [rn_mean.get(m, np.nan) for m in MODEL_ORDER],
})
split_cmp["inflation"] = split_cmp["random_mean"] - split_cmp["scaffold_mean"]
split_cmp.to_csv(f"{OUTDIR}/table09_partition_scheme.csv", index=False)
print()
print(split_cmp.round(4).to_string(index=False))
print(f"\nMean inflation across models: {split_cmp['inflation'].mean():+.4f}")

                                 comparison  n_pairs  mean_difference  median_difference  n_improved  n_worsened  wilcoxon_p  rank_biserial
                           GIN - GIN_noedge       15           0.0003            -0.0016           6           9      1.0000           0.00
                           GAT - GAT_noedge       15          -0.0011             0.0018           8           7      0.8904          -0.05
          random_XGBoost - scaffold_XGBoost       15           0.1174             0.1128          15           0      0.0001           1.00
random_RandomForest - scaffold_RandomForest       15           0.1026             0.1049          15           0      0.0001           1.00
                  random_GCN - scaffold_GCN       15           0.1091             0.1095          15           0      0.0001           1.00
                  random_GIN - scaffold_GIN       15           0.0987             0.0942          15           0      0.0001           1.00
                  ra

In [13]:
total = runs.groupby("arm")["wall_clock_min"].sum() / 60
print("Total computation by arm (hours)")
print(total.round(2).to_string())
print("\nArtefacts written to", OUTDIR)
for f in ["benchmark_runs.csv", "table06_scaffold_summary.csv", "table07_nemenyi.csv",
          "table07b_mean_ranks.csv", "table08_paired_comparisons.csv",
          "table09_partition_scheme.csv", "hpo_gnn_noedge.json"]:
    print("  -", f)

Total computation by arm (hours)
arm
ablation     2.61
main        11.13

Artefacts written to /content/drive/MyDrive/ames_pipeline/results
  - benchmark_runs.csv
  - table06_scaffold_summary.csv
  - table07_nemenyi.csv
  - table07b_mean_ranks.csv
  - table08_paired_comparisons.csv
  - table09_partition_scheme.csv
  - hpo_gnn_noedge.json


In [14]:
!cd /content/drive/MyDrive/ames_pipeline && zip -qr /content/part3_results.zip results
from google.colab import files
files.download("/content/part3_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>